# MangoLeaf AI V5.0.0: OE expansion from saved V4.5.0 weights

**Unexecuted experimental workflow; no adoption, deployment, accuracy or speed improvement is claimed.** Keep the original seven classes and frozen mango splits (1,854 train / 398 validation / 397 test). Start from the exact uncalibrated V4.5.0 stage-2 checkpoint with a fresh optimizer. This is warm-start fine-tuning, not an exact optimizer resume.

Expand approved OE from 2,011 to an expected **2,511 unique photos** by transferring the original 500 reviewed OOD photos. The 300 previously removed from raw OE are already inside those 500: do not add them twice. Verify current files and targeted cross-pool similarities; import the existing sealed scope review instead of reviewing all 500 again. All transferred 500 become training data and cannot report new-model OOD performance.

The default new schedule is five epochs at 1e-5 with frozen batch normalization and OE weight 0.5. It is an experimental starting choice, not a proven optimum. Epoch zero can be selected if no new epoch improves clean validation loss. Preserve V4.5.0 unchanged; every attempt writes a fresh V5.0.0 run.

**Upload both this notebook and its matching `v5_workflow.py` helper to Colab.** Put the helper at `/content/v5_workflow.py` using the Files panel. Do not install a historical environment freeze. Setup installs the explicit package plan, requires a restart, audits the installed dependency closure, and separately preflights the browser converter. Unknown/unreviewed audit findings stop execution. Existing narrowly reviewed converter packaging advisories remain visible, not patched.

This workflow does not change the PWA, its model files, or browser GPU precision settings.

## Step 1: Mount Drive, install the explicit runtime, then restart

Choose a Python 3.11/3.12 GPU runtime. Step 1 checks the actual Python version before installing packages. Execute this cell once. After installation, use **Runtime > Restart session**, then rerun this cell and continue. Runtime availability and GPU allocation are not guaranteed by the account subscription.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util

HELPER = Path('/content/v5_workflow.py')
EXPECTED_HELPER_SHA256 = '097278859dbfd8f3c64c5267a84aaa6937b462b4cca45dd9c6ce8aeb91bcc5e3'
if not HELPER.is_file():
    raise FileNotFoundError('Upload v5_workflow.py to the Colab Files panel (/content), beside this notebook workflow.')
if hashlib.sha256(HELPER.read_bytes()).hexdigest() != EXPECTED_HELPER_SHA256:
    raise RuntimeError('Helper differs from this notebook. Upload the matching repository copy of both files.')
spec = importlib.util.spec_from_file_location('mangoleaf_v5_workflow', HELPER)
wf = importlib.util.module_from_spec(spec)
spec.loader.exec_module(wf)
wf.install_runtime()


## Step 2: Verify imported versions, audit coverage and real GPU execution

Training dependencies must pass their entire installed closure audit. Provider-package conflicts remain visible separately. Converter setup below is a separate CPU subprocess environment with its own audit; neither audit proves a future Colab run or browser parity.

In [ ]:
runtime = wf.verify_runtime(require_gpu=True)


## Step 3: Select immutable inputs and a new V5.0.0 output

The parent path is your second-account saved training run. Keep its files unchanged. `OLD_OOD` is the original 500-photo folder, which can be a reviewed My Drive shortcut.

Set `PARENT_THRESHOLD_RUN` to the old threshold experiment folder (originally in your third account). Share that folder into this account and add a My Drive shortcut, or copy only these **unchanged sealed review files** into an input folder: `threshold_fitting_complete.json`, `evaluation_inputs.json`, `ood_inventory.json`, `ood_review.csv`, `similarity_review.csv`. Their recorded hashes are checked. A threshold run is not assumed to exist in this second account automatically. Missing review access stops before importing scope declarations; it does not require another full manual review of the 500.

`FRESH_OOD = None` allows training, calibration, paired mango figures and an experimental export with OOD explicitly pending. To report OOD later without retraining, use the separate V5.0.0 evaluation notebook and a genuinely unseen collection. A supplied fresh pool is reporting-only and must never affect checkpoint, temperature or gate selection.

Change the short schedule only before creating/freezing a new run. Never point an output at a prior run, input folder, or shared output shortcut.

In [ ]:
DRIVE = Path('/content/drive/MyDrive')
PARENT_RUN = DRIVE / 'mangoleaf_experiments/v4.5.0/training/v4-5-0-training-20261011T063913Z-40b7b0fe'
OLD_OOD = DRIVE / 'mango_ood_test_v4_4_0'
PARENT_THRESHOLD_RUN = None  # Set the actual My Drive path to the sealed five-file review bundle.
FRESH_OOD = None  # Optional: actual My Drive path to new, never-used unsupported photos.
OUTPUT_ROOT = DRIVE / 'mangoleaf_experiments/v5.0.0/training'
ALLOW_SHARED_INPUT_SHORTCUTS = True  # Only reviewed input shortcuts; outputs must belong to this account.

if PARENT_THRESHOLD_RUN is None:
    raise RuntimeError('Set PARENT_THRESHOLD_RUN to the old sealed review folder/bundle before creating a new run.')

experiment = wf.Experiment({
    'parent_run': str(PARENT_RUN), 'old_ood': str(OLD_OOD),
    'parent_threshold_run': str(PARENT_THRESHOLD_RUN),
    'fresh_ood': str(FRESH_OOD) if FRESH_OOD else None,
    'output_root': str(OUTPUT_ROOT), 'allow_shared_inputs': ALLOW_SHARED_INPUT_SHORTCUTS,
    'epochs': 5, 'learning_rate': 1e-5, 'batch_size': 16,
    'oe_loss_weight': 0.5, 'seed': 42, 'expected_oe_count': 2511,
})
print('Preserve this exact output:', experiment.run)


## Step 4: Verify photo bytes, preserved splits, existing scope review and new similarities

The helper verifies the exact parent checkpoint, class order, plan, image identities, splits and seven validation Grad-CAM originals. It references parent photos without copying another entire dataset into Drive. It checks decoded duplicates and perceptual cross-pool similarities in the expanded collection. These checks do not prove source independence.

Old review declarations and their original limitations are preserved, including previously evaluated versus new-for-V4.5.0 history and unknown individual sources/capture groups. Only newly flagged relationships need targeted review. New unsupported photos must not include supported-class mango specimens merely because a current model misclassifies them.

In [ ]:
experiment.inventory()
experiment.prepare_reviews()


## Step 5: Approve only targeted reviews and any fresh reporting collection

Open `cross_pool_similarity_review.csv` and corresponding `similarity_previews/`. For each retained cross-pool pair, use `decision=distinct` only after checking the actual photos; add a reason in `notes`. Related/uncertain images require a revised new run, rather than declaring them distinct to pass a guard. A header-only file means no candidates and needs no invented rows.

If a fresh OOD pool is supplied, complete `fresh_ood_review.csv`: `keep=yes/no`, retained `unsupported=yes`, known `captureGroup`/`sourceGroup`, honest `sourceNotes`, `priorUse=never_used_for_training_or_tuning`, and exclusion reasons in `notes`. Leave unknown capture/source groups empty and explain the limitation; never fabricate them. Excluded fresh photos do not need distinct-pair approval. All original 500 retain their matching old review; do not relabel them as fresh unseen OOD.

Edit the CSVs in the active run, then set the approval below. Do not recreate Step 3 merely to resume a review pause.

In [ ]:
REVIEW_COMPLETE = False
REVIEW_NOTES = ''  # Describe targeted decisions, actual provenance and remaining source/capture uncertainty.
experiment.approve(review_complete=REVIEW_COMPLETE, review_notes=REVIEW_NOTES)


## Step 6: Preflight the converter and inspect augmentation on the same originals

The converter creates a separate binary-wheel-only environment. It records exact package versions, full audit findings and architecture export proof before any fitting. The two exact legacy packaging-operation reviews remain unresolved and visible; no other finding is waived. If this preflight fails, preserve logs and diagnose before training.

The augmentation recipe matches V4.5.0: flips, rotation, zoom, shear, brightness and contrast. Inspect all seven examples to ensure specimen and disease details remain visible. Validation originals shown here are illustrations; augmentation is used only for training inputs.

In [ ]:
wf.prepare_converter(experiment.run)
experiment.build()
experiment.augmentation_preview()


## Step 7: Freeze the reviewed run before learning

A frozen plan binds exact parent identities, membership, review CSVs, settings, runtime evidence, helper bytes and preview. Subsequent edits require a new attempt. No V4.5.0 file is rewritten.

In [ ]:
AUGMENTATION_REVIEW_COMPLETE = False
AUGMENTATION_REVIEW_NOTES = ''
experiment.freeze(augmentation_review_complete=AUGMENTATION_REVIEW_COMPLETE,
                  augmentation_review_notes=AUGMENTATION_REVIEW_NOTES)


## Step 8: Fine-tune with a fresh Adam optimizer

The learned parent weights initialize the same architecture. Batch-normalization statistics remain frozen. Each epoch processes all 1,854 mango training photos; a continuous shuffled OE iterator carries remaining photos across epochs. `oe_exposure.json` records actual and selected-checkpoint coverage, rather than assuming every configurable schedule covers all 2,511.

Mango class weights derive only from training mango counts. OE uses a uniform target over seven existing outputs, never a fabricated eighth class. Checkpoint choice uses clean unweighted mango validation loss, including the epoch-zero parent baseline. OE loss is not diagnostic accuracy.

An interrupted fitting attempt cannot be resumed as if it had its optimizer state. Preserve its logs; create a new V5 attempt if fitting must restart. This shortcut reduces repeated learning work; no time or accuracy improvement is guaranteed.

In [ ]:
experiment.train()


## Step 9: Fit a fresh temperature on validation only and fold once

Calibration reloads the sealed uncalibrated selected model, fits only the 398 validation labels, and divides the logit head once. Neither mango test nor old/new OOD participates. The original V4.5.0 temperature is not reused or folded again.

In [ ]:
experiment.calibrate()


## Step 10: Measure fixed 70% confidence-only behavior once

No margin/entropy search is run. The gate accepts confidence **at least 0.70**, otherwise rejects. All 397 historical mango test photos remain in the denominator; rejected mango photos reduce recall. These same old test identities provide a paired exploratory reference, not a newly independent test set.

When fresh OOD exists, report its rejection and false acceptance without using its outcomes for tuning. Otherwise `freshOod=null` and its status remains pending. The transferred 500 must never appear in the new OOD report.

In [ ]:
experiment.evaluate()


## Step 11: Generate new training, mango metrics and Grad-CAM figures

Creates new `training_curves.png`, `augmentation_preview.png`, `grad_cam.png`, `confusion_matrix.png`, `per_class_metrics.png`, `confusion_matrix_with_rejection.png` and `per_class_metrics_with_rejection.png`. Curves show only the new fine-tuning epochs; they do not reuse the old five-plus-ten-epoch title.

Grad-CAM uses the same seven validation originals and the same true-class target as V4.5.0, with new selected weights. Calibrated labels/confidence remain visible. Highlighted areas may differ; the color scale remains consistent. Illustrations do not prove model correctness.

In [ ]:
experiment.figures()


## Step 12: Export the experimental float32 browser model

Exports the calibrated classifier without quantization and includes fixed confidence-only settings in `model_config.json`. `candidate_model.zip` contains browser model files. Missing unseen OOD or browser parity is explicitly pending. This does not install the model into MangoLeaf AI or change its GPU texture preference.

In [ ]:
experiment.export()


## Step 13: Seal evidence and read the limits

A complete OOD experiment requires a reviewed unseen pool and reporting receipt. Without it, this is a training handoff with pending OOD evidence. Browser numerical parity, actual devices, expert field evaluation and an explicit adoption decision remain separate. Save the exact run identity and hashes. Do not overwrite V4.5.0 or claim V5.0.0 is better merely because it trained.

If collecting fresh OOD later, open `MangoLeaf_AI_Evaluate_v5_0_0.ipynb`, point it at this sealed run and keep the model fixed. No additional training is needed for that evaluation.

In [ ]:
experiment.seal()


## Primary references

- [Keras transfer learning and fine-tuning](https://keras.io/guides/transfer_learning/)
- [Outlier Exposure paper](https://arxiv.org/abs/1812.04606)
- [Keras Grad-CAM](https://keras.io/examples/vision/grad_cam/)
- [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html)
- [TensorFlow.js converter](https://www.tensorflow.org/js/tutorials/conversion/import_saved_model)
- [Visible setuptools advisory](https://github.com/advisories/GHSA-h35f-9h28-mq5c)
- [Visible wheel advisory](https://github.com/advisories/GHSA-8rrh-rw8j-w5fx)
